<a href="https://colab.research.google.com/github/jeongnim078/NLP/blob/main/text_vectorization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

note:
Discussion point:
Identify the limitations of BoW: high dimensionality, sparsity, loss of word order, and absence of semantic relationships.
Describe the distributional hypothesis ("a word is known by the company it keeps") and how it motivates word embeddings.
Explain why static embeddings cannot handle polysemy or context, motivating the move to contextual models.
Compare BoW, Word2Vec, and BERT representations on the same text and articulate the tradeoffs (interpretability, compute cost, semantic richness) that guide which to use for a given social-science analysis.

# Text Vectorization: from BoW to Word2Vec to BERT

* * *

### This tutorial is adapted from
- https://github.com/dlab-berkeley/Python-NLP-Fundamentals
- https://github.com/practical-nlp/practical-nlp-code
- https://mccormickml.com/2019/05/14/BERT-word-embeddings-tutorial/

In the previous tutorial, we learned how to clean and preprocess text. But preprocessed text is still just text — and computers don't operate on words, they operate on numbers. Before we can do *any* computational analysis (clustering documents, classifying tweets, measuring similarity between speeches, training a model), we need a way to turn text into a **numeric representation**.

This tutorial walks through three landmark approaches to that problem, each one motivated by the limitations of the last:

*직접 다 실행해본 후 데이터셋에 알맞는 도구 선택하는 게 중요


1. **Bag-of-Words (BoW)** — the most straightforward representation: count which words appear in each document. Simple, interpretable, and still a strong baseline for many social-science tasks.

2. **Word2Vec** — learns *dense* word vectors from the contexts in which words appear, so that semantically related words end up close together in vector space.

3. **BERT** — produces *contextual* embeddings: the vector for "bank" in "river bank" differs from "bank" in "investment bank." This is the representation that powers most modern NLP.

By the end, you'll have hands-on experience with all three and a clearer sense of which one to reach for in your own analyses.

We'll lean on `scikit-learn` for BoW, `gensim` for Word2Vec, and `transformers` (Hugging Face) for BERT.

Let's install `scikit-learn` first!


In [ ]:
# Uncomment to install the package
# !pip install scikit-learn

In [ ]:
# Import other packages
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from string import punctuation
%matplotlib inline

In [ ]:
# Load data (for public sharing)
import os
import gdown

# Upload the data file to your Google Drive, and turn on the link sharing
# Replace 'YOUR_FILE_ID' with the actual file ID from your public Google Drive link e.g., "https://drive.google.com/file/d/YOUR_FILE_ID/view?usp=sharing"
file_id = '1qqIG8_-Y4JwbBj86aR_1hnaIJOkOjgxS'

# Name for the downloaded file in Colab
output_filename = 'airline_tweets.csv'

if file_id and not os.path.exists(output_filename):
    gdown.download(id=file_id, output=output_filename, quiet=False)
    print(f"File '{output_filename}' downloaded successfully!")
else:
    print(f"Using existing file '{output_filename}'.")

# Now you can load it with pandas
tweets= pd.read_csv(output_filename)
display(tweets.head())


As a refresher, each row in this dataframe correponds to a tweet. The following columns are of main interests to us. There are other columns containing metadata of the tweet, such as the author of the tweet, when it was created, the timezone of the user, and others, which we will set aside for now.

- `text` (`str`): the text of the tweet.
- `airline_sentiment` (`str`): the sentiment of the tweet, labeled as "neutral," "positive," or "negative."
- `airline` (`str`): the airline that is tweeted about.
- `retweet count` (`int`): how many times the tweet was retweeted.


# Apply a Text Cleaning Pipeline
Write a function called `preprocess` that performs the following steps on a text input:

* Step 1: Lowercase the text input.
* Step 2: Replace the following patterns with placeholders. ithout this, each unique URL or @handle becomes its own one-off column in the DTM ("vocab explosion")
    * URLs &rarr; ` URL `
    * Digits &rarr; ` DIGIT `
    * Hashtags &rarr; ` HASHTAG `
    * Tweet handles &rarr; ` USER `
* Step 3: Remove extra blankspace.

In [ ]:
import re

def preprocess(text):
    '''Create a preprocess pipeline that cleans the tweet data.'''

    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Replace patterns with placeholders
    def placeholder(text):
        # Regex patterns for URLs, digits, hashtags, and users
        url_pattern = r'(http|ftp|https):\/\/([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:\/~+#-]*[\w@?^=%&\/~+#-])'
        digit_pattern = r'\d+'
        hashtag_pattern = r'(?:^|\s)[＃#]{1}(\w+)'
        user_pattern = r'@(\w+)'

        # Replace URLs
        url_pattern = r'(http|ftp|https):\/\/([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:\/~+#-]*[\w@?^=%&\/~+#-])'
        url_repl = ' URL '
        text = re.sub(url_pattern, url_repl, text)
        # Replace digits
        digit_pattern = r'\d+'
        digit_repl = ' DIGIT '
        text = re.sub(digit_pattern, digit_repl, text)
        # Replace hashtags
        hashtag_pattern = r'(?:^|\s)[＃#]{1}(\w+)'
        hashtag_repl = ' HASHTAG '
        text = re.sub(hashtag_pattern, hashtag_repl, text)
        # Replace users
        user_pattern = r'@(\w+)'
        user_repl = ' USER '
        text = re.sub(user_pattern, user_repl, text)
        return text

    text = placeholder(text)

    # Step 3: Remove extra whitespace characters
    blankspace_pattern = r'\s+'
    blankspace_repl = ' '
    text = re.sub(blankspace_pattern, blankspace_repl, text)
    text = text.strip()

    return text


In [ ]:
# Test with example tweet first
example_tweet = 'lol @justinbeiber and @BillGates are like soo 2000 #yesterday #amiright saw it on https://twitter.com #yolo'

# Print the example tweet
print(example_tweet)
print(f"{'='*50}")

# Print the preprocessed tweet
print(preprocess(example_tweet))

In [ ]:
# Apply the function to the text column and assign the preprocessed tweets to a new column
tweets['text_processed'] = tweets['text'].apply(lambda x: preprocess(x))
tweets['text_processed'].head()

<a id='section3'></a>
# Part 1: The Bag-of-Words Representation

The idea of bag-of-words (BoW), as the name suggests, is quite intuitive: we take a document and toss it in a bag. The action of "throwing" the document in a bag disregards the relative position between words, so what is "in the bag" is essentially "an unsorted set of words" [(Jurafsky & Martin, 2024)](https://web.stanford.edu/~jurafsky/slp3/ed3book.pdf). In return, we have a list of unique words and the frequency of each of them.

For example, as shown in the following illustration, the word "coffee" appears twice.

<img src='https://github.com/ahnchive/NLP-for-SocialScience/blob/main/images/bow-illustration-1.png?raw=1' alt="BoW-Part2" width="600">


## Document Term Matrix

Now let's implement the idea of bag-of-words. Before we dive deeper, let's step back for a moment. In practice, text analysis often involves handling many documents; from now on, we use the term **document** to represent a piece of text on which we perform analysis. It could be a phrase, a sentence, a tweet, or any other text—as long as it can be represented by a string, the length dosen't really matter.

Imagine we have four documents (i.e., the four phrases shown above), and we toss them all in the bag. Instead of a word-frequency list, we'd expect a **document-term matrix (DTM)** in return. In a DTM, the word list is the **vocabulary** (V) that holds all unique words occur across the documents. For each **document** (D), we count the number of occurence of each word in the vocabulary, and then plug the number into the matrix. In other words, the DTM we will construct is a $D \times V$ matrix, where each row corresponds to a document, and each column corresponds to a token (or "term").

The unique tokens in this set of documents, arranged in alphabetical order, form the columns. For each document, we mark the occurence of each word present in the document. The numerical representation for each document is a row in the matrix. For example, the first document, "the coffee roaster," has the numerical representation $[0, 1, 0, 0, 0, 1, 1, 0]$.

Note that the left index column now displays these documents as text, but typically we would just assign an index to each of them.

$$
\begin{array}{c|cccccccccccc}
 & \text{americano} & \text{coffee} & \text{iced} & \text{light} & \text{roast} & \text{roaster} & \text{the} & \text{time} \\\hline
\text{the coffee roaster} &0 &1	&0	&0	&0	&1	&1	&0 \\
\text{light roast} &0 &0	&0	&1	&1	&0	&0	&0 \\
\text{iced americano} &1 &0	&1	&0	&0	&0	&0	&0 \\
\text{coffee time} &0 &1	&0	&0	&0	&0	&0	&1 \\
\end{array}
$$

To create a DTM, we will use `CountVectorizer` from the package `sklearn`. Look at the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) and see what options are available.  

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# A toy example containing four documents
test = ['the coffee roaster',
        'light roast',
        'iced americano',
        'coffee time']

# Create a CountVectorizer object
vectorizer = CountVectorizer() # For now we can just leave it blank to use the default settings.

# Fit and transform to create a DTM
test_count = vectorizer.fit_transform(test)

print(test_count)

Apparently we've got a "sparse matrix"—a matrix that contains a lot of zeros. This makes sense. For each document, there are words that don't occur at all, and these are counted as zero in the DTM. This sparse matrix is stored in a "Compressed Sparse Row" format, a memory-saving format designed for handling sparse matrices.

Let's convert it to a dense matrix, where those zeros are probably represented, as in a numpy array.

In [ ]:
# Convert DTM to a dense matrix
test_count.todense()

So this is our DTM! The matrix is the same as shown above. To make it more reader-friendly, let's convert it to a dataframe. The column names should be tokens in the vocabulary, which we can access with the `get_feature_names_out` function.

In [ ]:
# Create a DTM dataframe
print("column names: ", vectorizer.get_feature_names_out())
test_dtm = pd.DataFrame(data=test_count.todense(),
                        columns=vectorizer.get_feature_names_out())
display(test_dtm)


## DTM for Tweets

We'll begin by initializing a `CountVectorizer` object. In the following cell, we have included a few parameters that people often adjust. These parameters are currently set to their default values.

When we construct a DTM, the default is to lowercase the input text. If nothing is provided for `stop_words`, the default is to keep them. The next three parameters are used to control the size of the vocabulary, which we'll return to in a minute.

In [ ]:
# Create a CountVectorizer object
vectorizer = CountVectorizer(lowercase=True,
                             stop_words=None,
                             min_df=1,
                             max_df=1.0,
                             max_features=None)
# Fit and transform to create DTM
counts = vectorizer.fit_transform(tweets['text_processed'])
counts

In [ ]:
# Extract tokens
tokens = vectorizer.get_feature_names_out()
# Create DTM
first_dtm = pd.DataFrame(data=counts.todense(),
                         index=tweets.index,
                         columns=tokens)
print("DTM shape: ", first_dtm.shape)
first_dtm.head()



In [ ]:
# Most frequent tokens
first_dtm.sum().sort_values(ascending=False).head(10) # default axis=0 (over rows)

In [ ]:
# Which token appears most frequently in each tweet?
counts = pd.DataFrame()
counts['token'] = first_dtm.idxmax(axis=1) # axis=1 (over columns)

# Retrieve the number of occurrence
counts['number'] = first_dtm.max(axis=1)
counts


# Filter out placeholders
counts[(counts['token']!='digit')
       & (counts['token']!='hashtag')
       & (counts['token']!='user')].sort_values('number', ascending=False).head(10)

It looks like among all tweets, at most a token appears six times, and it is either the word "It" or the word "worst."

Let's go back to our tweets dataframe and locate the 918th tweet.

In [ ]:
# Retrieve 1214th tweet: "worst"
tweets.iloc[1214]['text']

## Customize the `CountVectorizer`

So far we've always used the default parameter setting to create our DTMs, but in many cases we may want to customize the `CountVectorizer` object. The purpose of doing so is to further filter out unnecessary tokens. In the example below, we tweak the following parameters:

- `stop_words = 'english'`: ignore English stop words
- `min_df = 2`: ignore words that don't occur at least twice
- `max_df = 0.95`: ignore words if they appear in more than 95\% of the documents

Oftentimes, we are not interested in words whose frequencies are either too low or too high, so we use `min_df` and `max_df` to filter them out. Alternatively, we can define our vocabulary size as $N$ by setting `max_features`. In other words, we tell `CountVectorizer` to only consider the top $N$ most frequent tokens when constructing the DTM.

In [ ]:
# Customize the parameter setting
vectorizer = CountVectorizer(lowercase=True,
                             stop_words='english',
                             min_df=2,
                             max_df=0.95,
                             max_features=None)
# Fit, transform, and get tokens
counts = vectorizer.fit_transform(tweets['text_processed'])
tokens = vectorizer.get_feature_names_out()

# Create the second DTM
second_dtm = pd.DataFrame(data=counts.todense(),
                          index=tweets.index,
                          columns=tokens)

print(first_dtm.shape)
print(second_dtm.shape)

Our second DTM has a substantially smaller vocabulary compared to the first one.

In [ ]:
second_dtm.sum().sort_values(ascending=False).head(10)

## Lemmatize the Text Input

So far, our DTM treats **every word form as a separate token**. That means "run", "runs", "running", and "ran" are all stored in different columns — even though, for most analyses, they refer to the same underlying concept.

This is wasteful in two ways:
1. It inflates the vocabulary (more columns, more sparsity).
2. It splits the signal: a topic about "running" gets diluted across four near-synonymous features instead of being concentrated in one.

The standard fix is to collapse word variants into a single canonical form. There are two common approaches:

### Stemming
Chops off suffixes using simple rules. Fast, but crude — it doesn't care whether the result is a real word.

```
running  → run
runs     → run
ran      → ran      ← stemmer can't handle irregular forms
studies  → studi    ← result isn't a real word
better   → better   ← can't relate to "good"
```


### Lemmatization
Looks up the dictionary base form (the "lemma") of each word, using part-of-speech information. Slower, but linguistically correct.

```
running  → run
ran      → run
studies  → study
better   → good
```

This requires a model that knows the language's morphology and vocabulary — which is exactly what `spaCy` provides.

Now let's implement lemmatization on our tweet data and use the lemmatized text to create a third DTM.



In [ ]:
# Import spaCy
import spacy
nlp = spacy.load('en_core_web_sm')

# Create a function to lemmatize text
def lemmatize_text(text):
    '''Lemmatize the text input with spaCy annotations.'''

    # Step 1: Initialize an empty list to hold lemmas
    lemma = []

    # Step 2: Apply the nlp pipeline to input text
    doc = nlp(text)

    # Step 3: Iterate over tokens in the text to get the token lemma
    for token in doc:
        lemma.append(token.lemma_)

    # Step 4: Join lemmas together into a single string
    text_lemma = ' '.join(lemma)

    return text_lemma

In [ ]:
# Apply the function to an example tweet
print(tweets.iloc[101]["text_processed"])
print(f"{'='*50}")
print(lemmatize_text(tweets.iloc[101]['text_processed']))

In [ ]:
# This may take a while!
tweets['text_lemmatized'] = tweets['text_processed'].apply(lambda x: lemmatize_text(x))

In [ ]:
# Print the preprocessed tweet
print(tweets['text_processed'].iloc[101])
print(f"{'='*50}")
# Print the lemmatized tweet
print(tweets['text_lemmatized'].iloc[101])

Now with the `text_lemmatized` column, let's create a third DTM. The parameter setting is the same as the second DTM.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# Create the vectorizer (the same param setting as previous)
vectorizer = CountVectorizer(lowercase=True,
                             stop_words='english',
                             min_df=2,
                             max_df=0.95,
                             max_features=None)

# Fit, transform, and get tokens
counts = vectorizer.fit_transform(tweets['text_lemmatized'])
tokens = vectorizer.get_feature_names_out()

# Create the third DTM
third_dtm = pd.DataFrame(data=counts.todense(),
                         index=tweets.index,
                         columns=tokens)

# Print the shapes of three DTMs
print(first_dtm.shape)
print(second_dtm.shape)
print(third_dtm.shape)

third_dtm.head()

Let's print the top 10 most frequent tokens as usual. These tokens are now lemmas and their counts also change after lemmatization.

In [ ]:
# Get the most frequent tokens in the third DTM
third_dtm.sum().sort_values(ascending=False).head(10)

In [ ]:
# Get the most frequent tokens in the third DTM
second_dtm.sum().sort_values(ascending=False).head(10)

<a id='section4'></a>

## Term Frequency-Inverse Document Frequency (TF-IDF)

So far, we're relying on word frequency to give us information about a document. This assumes if a word appears more often in a document, it's more informative. However, this may not always be the case. For example, we've already removed stop words because they are not informative, despite the fact that they appear many times in a document. We also know the word "flight" is among the most frequent words, but it is not that informative, because it appears in many documents. Since we're looking at airline tweets, we shouldn't be surprised to see the word "flight"!

To remedy this, we use a weighting scheme called **tf-idf (term frequency-inverse document frequency)**. The big idea behind tf-idf is to weight a word not just by its frequency within a document, but also by its frequency in one document relative to the remaining documents. So, when we construct the DTM, we will be assigning each term a **tf-idf score**. Specifically, term $t$ in document $d$ is assigned a tf-idf score as follows:

<img src='https://github.com/ahnchive/NLP-for-SocialScience/blob/main/images/tf-idf_finalized.png?raw=1' alt="TF-IDF" width="1200">

In essence, the tf-idf score of a word in a document is the product of two components: **term frequency (tf)** and **inverse document frequency (idf)**. The idf acts as a scaling factor. If a word occurs in all documents, then idf equals 1. No scaling will happen. But idf is typically greater than 1, which is the weight we assign to the word to make the tf-idf score higher, so as to highlight that the word is informative. In practice, we add 1 to both the denominator and numerator ("add-1 smooth") to prevent any issues with zero occurrences.

We can also create a tf-idf DTM using `sklearn`. We'll use a `TfidfVectorizer` this time:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a tfidf vectorizer
vectorizer = TfidfVectorizer(lowercase=True,
                             stop_words='english',
                             min_df=2,
                             max_df=0.95,
                             max_features=None)

# Fit and transform
tf_dtm = vectorizer.fit_transform(tweets['text_lemmatized'])

# Create a tf-idf dataframe
tfidf = pd.DataFrame(tf_dtm.todense(),
                     columns=vectorizer.get_feature_names_out(),
                     index=tweets.index)
tfidf.head()

### Interpret TF-IDF Values

Using the TF-IDF table, you can identify the most important token(s) for each document (a tweet in our case).

In [ ]:
# Get the most important tokens in the 1214th tweet
print(tweets.iloc[1214]['text_lemmatized'])
tfidf.iloc[1214].sort_values(ascending=False).head(10)

You can also look at which tokens are distinctive for a *group* of tweets — e.g., those with positive vs. negative valence.

To explore this, we'll gather the indices of all positive (and negative) tweets and compute the mean TF-IDF score of each token within each group. Tokens with a high mean score in one group but not the other are the ones characterizing that sentiment.


In [ ]:
# Complete the boolean masks
positive_index = tweets[tweets['airline_sentiment'] == 'positive'].index
negative_index = tweets[tweets['airline_sentiment'] == 'negative'].index

# Complete the following two lines
pos = tfidf.loc[positive_index].mean().sort_values(ascending=False).head(10) # axis=0, mean over documents
neg = tfidf.loc[negative_index].mean().sort_values(ascending=False).head(10)

# plot the top 10 terms with the highest mean tf-idf values for positive/negative tweets
pos.plot(kind='barh',
         xlim=(0, 0.18),
         color='cornflowerblue',
         title='Top 10 terms with the highest mean tf-idf values for positive tweets')
plt.show()

neg.plot(kind='barh',
         xlim=(0, 0.18),
         color='darksalmon',
         title='Top 10 terms with the highest mean tf-idf values for negative tweets')
plt.show()

# Part 2: Word Embeddings

In Part 1, we have tried converting the text data to a numerical representation with a Bags of Words (BoW) representation and Term Frequency-Inverse Document Frequeny (TF-IDF). These methods make heavy use of **word frequency** but not much of the relative position between words, but there's still rich semantic meanings and syntactic relations left to be captured beyond independent word frequency.

In Part 2, we will dive into word embeddings, a method widely combined with more advanced Natural Language Processing (NLP) tasks. We'll make extensive use of the `gensim` package, which hosts a range of word embedding models, including `word2vec` and `GloVe`, the two models we'll explore today.


##  Understand Word Emebeddings

As famously put by British Linguist J.R. Firth:

> **You shall know a word by the company it keeps.**

This quote sums it all for the essence of word embeddings, which take the numerical representation of text further to the next step.

Recall from Part ` that a BoW representation is a **sparse** matrix. Its dimension is determined by vocabulary size and the number of documents. Importantly, a sparse matrix like BoW is interpretable; the cell values refer to the count of a word in a document. Oftentimes the cell values are zero: many words simply don't appear in a document at all.

We can think of word embedding as a matrix likewise, but this time a **dense** matrix, where the cell values are real numbers. Word embeddings project a word's meaning onto a high-dimensional vector space—that's why it is also called **word vectors**. A word vector is essentially an array of real numbers, the length of which, as we'll see today, could be as low as 50, or as high as 300 (or even higher in Large Language Models). These real numbers do not make explicit sense to us, but this is not to say they are *meaningless*. It is now believed that a word's meaning can be captured by the vector representation, which we will return to shortly.  

BOW:
- Sparse matrix
- Dimension: $D$ x $V$, where rows are **D**ocuments and columns are words in the **V**ocabulary.
- Interpretable: e.g., "bank" and "banker" may appear in a financial document, but probably not "bane."
  
<img src='https://github.com/ahnchive/NLP-for-SocialScience/blob/main/images/bow-illustration-2.png?raw=1' alt="BoW" width="500">

Word embeddings:
- Dense matrix
- Dimension: $V$ x $D$, where rows are **V**ocabulary and columns are vectors with dimension **D**.
- Not immediately interpretable

<img src='https://github.com/ahnchive/NLP-for-SocialScience/blob/main/images/bow-illustration-3.png?raw=1' alt="BoW" width="500">

Today, we are going to explore two widely used word embedding models, `word2vec` and `GloVe`. We will use the package `gensim` to access both models.


### Reading scientific notation

You'll notice embedding values are printed in a form like `2.5e-02` or `1.3e-01`. This is *scientific notation* — a compact way to write very small or very large numbers.

The `e` means "× 10 to the power of". The number after `e` tells you how many places to move the decimal point — left if negative, right if positive:

- 2.5e-02  =  2.5 × 10⁻²  =  0.025
- 1.3e-01  =  1.3 × 10⁻¹  =  0.13
- 4.7e+00  =  4.7 × 10⁰   =  4.7
- 8.0e+03  =  8.0 × 10³   =  8000


In [ ]:
# Uncomment to install gensim
# !pip install gensim

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gensim
import gensim.downloader as api
from gensim.models import KeyedVectors

## `word2vec`

The idea of word vectors, i.e, projecting a word's meaning onto a vector space, has been around for a long time. The `word2vec` model, proposed by [Mikolov et al.](https://arxiv.org/abs/1301.3781) in 2013, introduces an efficient model of word embeddings, since then it has stimulated a new wave of research on this topic.

The key question asked in this paper is: how do we go about learning a good vector representation from the data?

Mikolov et al. proposed two approaches: the **continuous bag-of-words (CBOW)** and the **skip-gram (SG)**. They are similar in that we use the vector representation of a token to try and predict what the nearby tokens are with a shallow neural network.   

Take the following sentence from Merriam-Webster for example. If our target token is $w_t$, "banks," the context tokens would be the preceding tokens $w_{t-2}, w_{t-1}$ and the following ones $w_{t+1}, w_{t+2}$. This example corresponds to a **window size** of 2: 2 words on either side of the target word. Similarly when we move onto the next tagret token, the context window moves as well.

<img src='https://github.com/ahnchive/NLP-for-SocialScience/blob/main/images/target_word.png?raw=1' alt="Trget word" width="500">

In the continuous bag-of-words model, our goal is to predict the target token, given the context tokens. In the skip-gram model, the goal is to predict the context tokens from the target token. This is the reverse of the continuous bag-of-words and is a harder task, since we have to predict more from less information.

<img src='https://github.com/ahnchive/NLP-for-SocialScience/blob/main/images/word2vec-model.png?raw=1' alt="word2vec" width="550">

**CBOW** (Left):
- **Input**: context tokens
- **Inner dimension**: embedding layer
- **Output**: the target token

**Skip-gram** (Right):
- **Input**: the target token
- **Inner dimension**: embedding layer
- **Output**: context tokens

The above figure illustrates the direction of prediction. It also serves as a schematic representation of a neural network, i.e., the mechanism underlying the training of `word2vec`. The input and output are known to us, represented by **one-hot encodings** in Mikolov et al. The **hidden layer**, the inner dimension between the input and the output, is the vector representation that the model is trying to learn.

Here, we provide a brief explanation of where embedding come from, but we won't go into the specifics of training a neural network. The `word2vec` model we will be interacting with today is **pretrained**, meaning that the embeddings have already been trained on a large corpus (or a number of corpora). Pretrained `word2vec` and `GloVe`, as well as other models, are available through `gensim`.

Let's take a look at the list of models that `gensim` provides:

In [ ]:
# Get word embedding models
gensim_models = list(api.info()['models'].keys())

for model in gensim_models:
    print(model)

These models are named following the **model-corpora-dimension** fashion. The one called `word2vec-google-news-300` is what we are looking for! This is a `word2vec` model that is trained on Google News; the dimension of word embedding is 300.

In [ ]:
# Run the following line if your local machine has plenty of memory
wv = api.load("word2vec-google-news-300")

# Alternatively, download the model to your machine and load it in
# Download link: https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?resourcekey=0-wjGZdNAUop6WykTtMip30g
# wv = KeyedVectors.load_word2vec_format('../data/GoogleNews-vectors-negative300.bin', binary=True)

In [ ]:
wv['banana']

## Word Similarity

The first question we can ask is: what words are similar to "bank"? In vector space, we expect similar words to have vectors that are close to each other.

There are many metrics for measuring vector similarity; one of the most widely used is [**cosine similarity**](https://en.wikipedia.org/wiki/Cosine_similarity). Orthogonal vectors have a cosine similarity of 0, and parallel vectors have a cosine similarity of 1.

<img src='https://github.com/ahnchive/NLP-for-SocialScience/blob/main/images/cosine-similarity.png?raw=1' alt="Trget word" width="500">


Using `word2vec`, we can ask the model to return the cosine similarity between two words by calling the function `similarity`. Let's go ahead and check out the similarities between the following four pairs of words. In each pair, we have "river" and "bank," but the form of "bank" varies. Let's see if word forms make a difference!

In [ ]:
# bank with capitalized B
wv.similarity('river', 'Bank')

In [ ]:
# the present participle
wv.similarity('river', 'banking')

In [ ]:
# the word stem
wv.similarity('river', 'bank')

In [ ]:
# the plural form
wv.similarity('river', 'banks')

`gensim` also provides a function called `most_similar` that lets us find the words most similar to a queried word. The output is a tuple containing candidate words and their cosine similarities to the queried word.


In [ ]:
wv.most_similar(['bank'])

##  Word Analogy

In addition to similarities between word pairs, one of the best known ways to use word vectors provided by `word2vec` is to solve word analogies. For example, consider the following analogy:

`man : king :: woman : ?`

Oftentimes, word analogy like this is visualized with parallelogram, such as shown in the following figure, which is adapted from [Ethayarajh et al. (2019)](https://arxiv.org/abs/1810.04882).


<img src='https://github.com/ahnchive/NLP-for-SocialScience/blob/main/images/word_analogy.png?raw=1' alt="Word analogy" width="500">


The upper side (difference between `man` and `woman`) should approximate the lower side (difference between `king` and `queen`); the vector difference represents the meanig of `female`.

- $\mathbf{V}_{\text{man}} - \mathbf{V}_{\text{woman}} \approx \mathbf{V}_{\text{king}} - \mathbf{V}_{\text{queen}}$

Similarly, the left side (difference between `king` and `man`) should approximate the right side (difference between `queen` and `woman`); the vector difference represents the meaning of `royal`.

- $\mathbf{V}_{\text{king}} - \mathbf{V}_{\text{man}} \approx \mathbf{V}_{\text{queen}} - \mathbf{V}_{\text{woman}}$

We can take either equation and rearrange it:

- $\mathbf{V}_{\text{king}} - \mathbf{V}_{\text{man}} + \mathbf{V}_{\text{woman}} \approx \mathbf{V}_{\text{Queen}}$

If the vectors of `king`, `man`, and `woman` are known, we can use simple vector arithmetic to obtain a vector that approximates the meaning of `queen`. This is the idea behind solving word analogies with word embeddings.

Let's implement it!

In the following operations, we use the function [`get_vector`](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.get_vector) to obtain a vector for a word. The function works similarly to using a key to access its value in the `word2vec` dictionary, but it also allows us to specify whether we want to normalize the vector.

We want to set `norm` to `True` and renormalize the difference vector later. This is because different vectors have different lengths; normalization puts everything on a common scale (unit vector)

In [ ]:
# Calculate the vector difference for "royal"
difference = wv.get_vector('king', norm=True) - wv.get_vector('man', norm=True)

# Add the vector for "woman"
woman_plus_difference = wv.get_vector('woman', norm=True) + difference

# Renormalize
woman_plus_difference = woman_plus_difference / np.linalg.norm(woman_plus_difference) # computes the L2 norm (Euclidean length) of the vector

In [ ]:
# What is the most similar vector?
wv.most_similar(woman_plus_difference)

In [ ]:
# Words that are similar to woman and king but dissimilar to man
wv.most_similar(positive=['woman', 'king'], negative='man')

### Woman is to Homemaker example

[Bolukbasi et al. (2016)](https://arxiv.org/pdf/1607.06520) is a thorough investigation of gender bias present in word embeddings, and they primarily focus on word analogies, especially those that reveal gender stereotyping! Let run a couple examples discussed in the paper, using the `most_similiar` function we've just learned.

The following code block contains a few examples we can pass to the `positive` argument: we want the output to be similar to, for example, `woman` and `chairman`, and in the meantime, we are also specificying that it should be dissimilar to `man`. We'll print the top result by indexing to the 0th item.

Let's complete the following for loop.

In [ ]:
positive_pair = [['woman', 'chairman'],
                 ['woman', 'doctor'],
                 ['woman', 'computer_programmer']]
negative_word = 'man'

In [ ]:
# Get the most similar word given positive and negative examples
for example in positive_pair:
    result = wv.most_similar(positive=example, negative=negative_word)
    print(f"man is to {example[1]} as woman is to {result[0][0]}")

## More models

Word2Vec is just one of several popular static word-embedding models. Two other widely used ones are **GloVe** and **fastText**. They all produce dense word vectors, but they get there in different ways.

A short way to think about each:

- *Word2Vec* learns each word as its own vector by predicting the words that surround it. Strong on local relationships.
- *GloVe* builds a giant word co-occurrence matrix (how often does word A appear near word B *across the entire corpus*?), then factorizes it. Captures more global structure.
- *fastText* is like Word2Vec, but each word is represented as the sum of vectors of its character n-grams. So "teach", "teacher", and "teaching" share most of their representation — and it can even produce a vector for a word it has never seen before.


In [ ]:
# Load GloVe (trained on Wikipedia + Gigaword, 300-dim)
glove = api.load("glove-wiki-gigaword-300")

In [ ]:
# GloVe: same queries we ran on Word2Vec
print("river–bank similarity:", glove.similarity('river', 'bank'))
print()
print("Most similar to 'bank':")
for w, s in glove.most_similar('bank')[:5]:
    print(f"  {w:<15} {s:.3f}")
print()
print("king - man + woman ≈ ?")
for w, s in glove.most_similar(positive=['woman', 'king'], negative=['man'])[:5]:
    print(f"  {w:<15} {s:.3f}")

### fastText and unseen words

fastText shares Word2Vec's training objective, but with one crucial twist: each word is represented as a bag of *character n-grams*.

For example, with n-grams of length 3–6, the word `teaching` becomes pieces like:

```
3-grams: <te, tea, eac, ach, chi, hin, ing, ng>
4-grams: <tea, teac, each, achi, chin, hing, ing>
5-grams: <teac, teach, eachi, achin, ching, hing>
6-grams: <teach, teachi, eachin, aching, ching>
plus the whole word: teaching
```

(The `<` and `>` mark word boundaries, so the model can distinguish prefixes/suffixes from infixes.)

During training, fastText learns a vector for **each n-gram**. The vector for a whole word is then just the **sum of the vectors of its n-grams**.

Why does this help?

1. *Morphology comes for free.* "teach", "teacher", "teaching", "teaches" share many of the same n-grams, so their vectors end up close together — without the model having to learn that fact from co-occurrence alone.
2. *Out-of-vocabulary (OOV) words still get a vector.* Even if fastText has never seen the word `teachable`, it has seen pieces like `<te`, `tea`, `each`, `able`, `ble>` during training. It can compose a reasonable vector from those subwords.

Caution!! To see real fastText OOV behavior, you'd need to:
1. Download the original `.bin` file from https://fasttext.cc/docs/en/english-vectors.html
2. Load it with `gensim.models.fasttext.load_facebook_vectors('cc.en.300.bin')`
3. Then `ft[oov_word]` and `ft.most_similar(oov_word)` will work.

We'll skip that here because the .bin file is ~7GB.

In [ ]:
# Load fastText (trained on Wikipedia + news, 300-dim, with subwords)
ft = api.load("fasttext-wiki-news-subwords-300")

In [ ]:
# fastText: same queries
print("river–bank similarity:", ft.similarity('river', 'bank'))
print()
print("Most similar to 'bank':")
for w, s in ft.most_similar('bank')[:5]:
    print(f"  {w:<15} {s:.3f}")
print()
print("king - man + woman ≈ ?")
for w, s in ft.most_similar(positive=['woman', 'king'], negative=['man'])[:5]:
    print(f"  {w:<15} {s:.3f}")

In [ ]:
# OOV demonstration with fastText
from gensim.models import FastText

toy_corpus = [
    ["teach", "the", "student", "math"],
    ["teacher", "explains", "the", "topic"],
    ["teaching", "is", "a", "skill"],
    ["students", "are", "learning"],
    ["a", "lesson", "in", "math"],
]

mini_ft = FastText(sentences=toy_corpus, vector_size=50, min_count=1,
                   min_n=3, max_n=6, epochs=20) # fasttext training

print("teachable" in mini_ft.wv.key_to_index)   # False
vec = mini_ft.wv["teachable"]                    # OOV works!
print(vec)
print(vec.shape)                                 # (50,)


### Which one is "better"?

There's no universal winner — it depends on the task:

| If you...                                              | Reach for...     |
|--------------------------------------------------------|------------------|
| Have a clean, well-known vocabulary (formal text)      | Word2Vec or GloVe |
| Care about morphology or word forms                    | fastText         |
| Work with noisy text, typos, or rare/technical terms   | fastText         |
| Analyze social media, multilingual data, code mixing   | fastText         |
| Want results that match a commonly cited baseline      | Word2Vec or GloVe |

For most modern social-science analyses of messy real-world text (tweets, user comments, news from many sources), fastText's OOV ability is a meaningful practical advantage.

## 한국어 모델

In [ ]:
from huggingface_hub import hf_hub_download
import fasttext

model_path = hf_hub_download(repo_id="facebook/fasttext-ko-vectors", filename="model.bin")
ft_ko = fasttext.load_model(model_path)

# 사용법이 약간 다름
ft_ko.get_nearest_neighbors('학생')
# → [(0.78, '학생들'), (0.71, '학교'), ...]

ft_ko.get_word_vector('학생').shape   # (300,)

# Part 3: BERT — Contextual Word Embeddings

So far, every embedding we've seen has been *static*: each word has exactly **one** vector, no matter where it shows up. That means "bank" in *"river bank"* and "bank" in *"bank robber"* get the same Word2Vec vector — even though they mean completely different things.

BERT (Bidirectional Encoder Representations from Transformers, Devlin et al. 2018) changes the game. Instead of looking up a precomputed vector, BERT *generates* a vector for each word **on the fly**, conditioned on the full sentence around it. So:

```
"After stealing money from the bank vault..."   →  vector for 'bank' #1
"...the bank robber was seen..."                →  vector for 'bank' #2
"...fishing on the Mississippi river bank."     →  vector for 'bank' #3
```

All three are the same string `bank`, but BERT produces *three different vectors*. We'll see in a moment that the first two (financial sense) end up much closer to each other than to the third (geographical sense).


In [ ]:
# Install Hugging Face's transformers library (plus PyTorch, which it depends on)
# !pip install transformers torch

In [ ]:
import torch
from transformers import BertTokenizer, BertModel

# Load the pretrained tokenizer (vocabulary + the WordPiece algorithm)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

## Input formatting

In [ ]:
# Try tokenizing a sentence that contains a rare word
text = "Here is the sentence I want embeddings for."
tokens = tokenizer.tokenize(text)
print(tokens)

# Our example sentence — three occurrences of "bank" with two different meanings
text = ("After stealing money from the bank vault, the bank robber was seen "
        "fishing on the Mississippi river bank.")
marked_text = "[CLS] " + text + " [SEP]"

# Tokenize, then map tokens → BERT vocab indices
tokenized_text = tokenizer.tokenize(marked_text)
indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)

# Print tokens with their IDs
for tok, idx in zip(tokenized_text, indexed_tokens):
    print(f"  {tok:<12} {idx:>6}")

## Running BERT and inspecting the output

Now we load the actual BERT model and run our tokenized sentence through it. We set `output_hidden_states=True` so we can see the **hidden state from every layer**, not just the final one.

A few things to know:
- `bert-base-uncased` has *12 transformer layers* + an initial embedding layer = **13 hidden states**.
- Each token gets a *768-dimensional vector* at each layer.
- `model.eval()` switches off training-only behavior (like dropout).
- `torch.no_grad()` tells PyTorch we're not training — saves memory and is faster.

In [ ]:
# Wrap our lists as PyTorch tensors and add a batch dimension
tokens_tensor = torch.tensor([indexed_tokens])

# Mark every token as belonging to sentence 1 (single-sentence input)
segments_ids = [1] * len(tokenized_text)
segments_tensors = torch.tensor([segments_ids])

# Load BERT and put it in evaluation mode
model = BertModel.from_pretrained('bert-base-uncased',
                                  output_hidden_states=True)
model.eval()

# Forward pass — no gradients needed since we're not training
with torch.no_grad():
    outputs = model(tokens_tensor, segments_tensors)

# hidden_states is a tuple of 13 tensors, each of shape (batch, n_tokens, 768)
hidden_states = outputs.hidden_states

print(f"Number of layers:        {len(hidden_states)}  (1 input embedding + 12 BERT layers)")
print(f"Shape per layer:         {tuple(hidden_states[0].shape)}  (batch, tokens, hidden_size)")

## Building word vectors from hidden states

Each token now has **13 different vectors** (one per layer). To get a single vector per token, we need to combine them somehow.

The original BERT paper experimented with several strategies. Two popular choices:

1. *Sum the last 4 layers* — gives one 768-d vector per token. Compact, captures contextual info from the late layers.
2. *Concatenate the last 4 layers* — gives one 4×768 = 3072-d vector per token. Larger, slightly more expressive on some tasks.

There's no universal best; the BERT authors found concatenation slightly better but both work well. We'll use **sum of the last 4 layers** here for simplicity.

We'll also restructure the tensor so the first dimension is *tokens* (instead of layers), since we want one vector per token.

In [ ]:
# hidden_states is a tuple of 13 tensors with shape (1, n_tokens, 768).
# Stack into a single tensor: shape (13, 1, n_tokens, 768)
token_embeddings = torch.stack(hidden_states, dim=0)
print("Stacked shape:", tuple(token_embeddings.shape), " (layers, batch, tokens, hidden_size)")

# Drop the batch dim → (13, n_tokens, 768)
token_embeddings = torch.squeeze(token_embeddings, dim=1)
print("Squeezed shape:", tuple(token_embeddings.shape), " (layers, tokens, hidden_size)")

# Swap layer and token dims → (n_tokens, 13, 768)
token_embeddings = token_embeddings.permute(1, 0, 2)
print("Reshaped to:", tuple(token_embeddings.shape), " (tokens, layers, hidden_size)")

# Build one 768-d vector per token by summing the last 4 layers
token_vecs_sum = [torch.sum(token[-4:], dim=0) for token in token_embeddings]
print(f"Got {len(token_vecs_sum)} token vectors, each of length {len(token_vecs_sum[0])}.")

## Confirming that the vectors are contextual

Let's go back to our three instances of "bank":

```
After stealing money from the [bank] vault,        ← position 6, financial
the [bank] robber was seen ...                     ← position 10, financial
... fishing on the Mississippi river [bank].       ← position 19, geographical
```

If BERT is really producing context-dependent vectors, we'd expect:
- *bank vault* ↔ *bank robber* — similar (both financial sense)
- *bank robber* ↔ *river bank* — different (different meanings)

We'll measure with **cosine similarity** (the same metric we used for Word2Vec).

In [ ]:
from scipy.spatial.distance import cosine

# Locate the three "bank" tokens in our tokenized sentence
print("Token positions:")
for i, tok in enumerate(tokenized_text):
    if tok == 'bank':
        print(f"  position {i:>2}: {tok}  (context: ...{' '.join(tokenized_text[max(0,i-2):i+3])}...)")

# Cosine similarity = 1 - cosine distance
sim1 = 1 - cosine(token_vecs_sum[6], token_vecs_sum[10])   # bank vault ↔ bank robber
sim2 = 1 - cosine(token_vecs_sum[10], token_vecs_sum[19])  # bank robber ↔ river bank

print()
print(f"Cosine similarity btw bank vault ↔ bank robber):    {sim1:.3f}")
print(f"Cosine similarity btw bank robber ↔ river bank):    {sim2:.3f}")

## 한국어로 바꾸려면 tokenizer와 model을 교체
```python
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained('klue/bert-base')
model = AutoModel.from_pretrained('klue/bert-base', output_hidden_states=True)
```

# Visualizing and Clustering Word Embeddings

We've spent the last few sections looking at word vectors *one query at a time* — checking similarity between word pairs, finding nearest neighbors, doing analogies. But each Word2Vec vector lives in **300-dimensional space**, which is impossible to look at directly. This is where **dimensionality reduction** techniques come in: they compress high-dimensional vectors down to 2 or 3 dimensions while preserving as much of the original structure as possible, letting us see the **shape of the semantic space** with our eyes.


In [ ]:
# Pick a few semantic categories and grab vectors for each word
word_groups = {
    'animals':   ['dog', 'cat', 'horse', 'cow', 'lion', 'tiger', 'elephant'],
    'fruits':    ['apple', 'banana', 'orange', 'grape', 'lemon', 'peach', 'mango'],
    'countries': ['japan', 'france', 'germany', 'italy', 'spain', 'china', 'brazil'],
    'sports':    ['soccer', 'basketball', 'baseball', 'tennis', 'golf', 'hockey', 'swimming'],
    'vehicles':  ['car', 'truck', 'bus', 'train', 'airplane', 'bicycle', 'motorcycle'],
}

# Flatten to a (word, category) list, keeping only words that exist in Word2Vec's vocabulary
words, labels = [], []
for category, ws in word_groups.items():
    for w in ws:
        if w in wv.key_to_index:
            words.append(w)
            labels.append(category)

# Stack their 300-d vectors into a matrix: shape (N_words, 300)
X = np.vstack([wv[w] for w in words])
print(f"Matrix shape: {X.shape}   ({len(words)} words × 300 dims)")
print(f"Categories:  {sorted(set(labels))}")

## Dimensionality reduction: PCA vs. t-SNE

We can't visualize 300 dimensions, so we need to **project** the vectors down to 2D. Two common approaches:

### PCA (Principal Component Analysis)
Finds the 2 directions in the original 300-d space along which the data varies the most, and projects everything onto them. *Linear* and *deterministic* — fast and easy to interpret, but can squash interesting non-linear structure flat.

### t-SNE (t-Distributed Stochastic Neighbor Embedding)
A *non-linear* method that tries to preserve **local** structure: points that are close in 300-d should remain close in 2D. Great for revealing clusters, but distances between far-apart clusters in the t-SNE plot don't mean much, and it's stochastic (different `random_state` → different layouts).

Rule of thumb:
- *PCA*: when you want to see overall global structure / variance directions.
- *t-SNE*: when you want to see whether discrete clusters exist.
- *UMAP*: similar to t-SNE, often faster, sometimes preserves global structure better. A common modern alternative.

In [ ]:
from sklearn.decomposition import PCA

# Project 300-d → 2-d
pca = PCA(n_components=2, random_state=0)
X_pca = pca.fit_transform(X)

# Plot
plt.figure(figsize=(10, 7))
categories = sorted(set(labels))
colors = plt.cm.tab10(np.linspace(0, 1, len(categories)))

for cat, color in zip(categories, colors):
    idx = [i for i, l in enumerate(labels) if l == cat]
    plt.scatter(X_pca[idx, 0], X_pca[idx, 1], color=color, label=cat, s=80, alpha=0.8)

for i, w in enumerate(words):
    plt.annotate(w, (X_pca[i, 0], X_pca[i, 1]), fontsize=9,
                 xytext=(4, 4), textcoords='offset points')

plt.title("PCA projection of Word2Vec vectors (300-d → 2-d)")
plt.xlabel(f"PC 1  (explains {pca.explained_variance_ratio_[0]:.1%} of variance)")
plt.ylabel(f"PC 2  (explains {pca.explained_variance_ratio_[1]:.1%} of variance)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.manifold import TSNE

# perplexity ≈ how many neighbors each point should consider (rule of thumb: 5–50;
# with small N use a smaller value)
tsne = TSNE(n_components=2, perplexity=5, random_state=0, init='pca')
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(10, 7))
for cat, color in zip(categories, colors):
    idx = [i for i, l in enumerate(labels) if l == cat]
    plt.scatter(X_tsne[idx, 0], X_tsne[idx, 1], color=color, label=cat, s=80, alpha=0.8)

for i, w in enumerate(words):
    plt.annotate(w, (X_tsne[i, 0], X_tsne[i, 1]), fontsize=9,
                 xytext=(4, 4), textcoords='offset points')

plt.title("t-SNE projection of Word2Vec vectors")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()